In [ ]:
import os
import sys

import pandas as pd
import yaml

from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.oasisb.oasisb_utils import (
    EM_EDAX_MIME_TYPES_SIDECAR,  # edax
    EM_EDAX_MIME_TYPES_SOLITARY,
    # APM_MIME_TYPES_SIDECAR,  # only for cross-referencing between apm and em collections
    # APM_MIME_TYPES_SOLITARY,
    EM_HFIVE_MIME_TYPES_SIDECAR,  # hdf
    EM_HFIVE_MIME_TYPES_SOLITARY,
    EM_IMAGE_MIME_TYPES_SIDECAR,  # image
    EM_IMAGE_MIME_TYPES_SOLITARY,
    EM_KPY_MIME_TYPES_SIDECAR,  # kikuchipy diffraction pattern
    EM_KPY_MIME_TYPES_SOLITARY,
    EM_MIXED_MIME_TYPES_SIDECAR,  # mixed, spectrum, etc.
    EM_MIXED_MIME_TYPES_SOLITARY,
    EM_MTEX_MIME_TYPES_SIDECAR,  # mtex
    EM_MTEX_MIME_TYPES_SOLITARY,
    get_project_id,
    prepare_parsing,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)

## Decompress the original files from the scientists from the storage location

Locally, original research data are stored compressed when not needed.<br>
Maybe multiple compressed files per project directory.<br>

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (1, 880)

with open(f"{src_directory}{os.sep}aaa_em_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

# keep_searching_toggle = True
count: int = 0  # how many files to decompress
volume: int = 0  # how much byte volume does this add to scratch
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.project_name)
        if project_range[0] <= int(project_id) <= project_range[1]:
            if row.legal == "1" and row.use == "1":
                if project_id in nomad_project_names:
                    status = prepare_parsing(
                        f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
                        src_directory,
                        project_id,
                        trg_directory,
                        report=True,
                        write=False,
                        mime_type="image",  # "image",  # "mtex", "hdf", "image", "mixed"
                        mime_type_solitary=EM_IMAGE_MIME_TYPES_SOLITARY,
                        mime_type_sidecar=EM_IMAGE_MIME_TYPES_SIDECAR,
                    )
                    for key, obj in status.items():
                        if obj["n"] > 0:
                            print(f"{project_id}, {key}, {obj['n']}, {obj['bytes']}")
                            count += obj["n"]
                            volume += obj["bytes"]
print(f"Batch queue completed, {count} files, {volume / 1024**3} GiB uncompressed")

***

Programmatic identification of atom_types from file names.

In [ ]:
pattern = os.path.join(
    f"{trg_directory}{os.sep}049.mixed.decompressed.csv", f".mixed.decompressed.csv"
)
decompression_logfiles: list[str] = glob.glob(pattern)
for file in decompression_logfiles:
    print(file)